In [1]:
# version 7 2025.12.04 ~ 05

import torch
print(torch.cuda.is_available())  # True면 정상
print(torch.cuda.get_device_name(0))  # GPU 이름 출력


True
NVIDIA GeForce GTX 1660 SUPER


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: NVIDIA GeForce GTX 1660 SUPER


In [3]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
import re
import warnings
from tqdm import tqdm
from gensim.models import FastText

warnings.filterwarnings("ignore")

from pycaret.regression import *
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error, mean_squared_log_error
)


class MercariPyCaretAnalyzer:

    def __init__(self, data_dir="../data", results_dir="../results", model_dir="../models"):

        self.data_dir = data_dir
        self.results_dir = results_dir
        self.model_dir = model_dir

        self.train = None
        self.test = None
        self.train_vectorized = None
        self.test_vectorized = None
        self.best_model = None
        self.setup_result = None
        self.metrics = {}

        os.makedirs(self.results_dir, exist_ok=True)
        os.makedirs(self.model_dir, exist_ok=True)


    def _safe_name(self, text):
        return re.sub(r"[^A-Za-z0-9_\-]", "", text)


    def load_data(self, train_file="train.tsv", test_file="test.tsv", sep="\t", undersample_frac=0.35):
        print("📂 Loading dataset...")

        self.train = pd.read_csv(os.path.join(self.data_dir, train_file), sep=sep)
        self.test = pd.read_csv(os.path.join(self.data_dir, test_file), sep=sep)

        self.train = self.train[self.train["price"] > 0].dropna(subset=["price"])
        self.train["price"] = np.log1p(self.train["price"])

        if undersample_frac:
            self._stratified_sample(frac=undersample_frac)

        for df in [self.train, self.test]:
            df["main_cat"], df["sub_cat"], df["sub_sub_cat"] = zip(
                *df["category_name"].apply(
                    lambda x: (x.split("/") if isinstance(x, str) and "/" in x else ["missing"] * 3)
                )
            )
            df["brand_name"] = df["brand_name"].fillna("Unknown").astype(str)
            df["item_description"] = df["item_description"].fillna("No description").astype(str)
            df["name"] = df["name"].fillna("No name").astype(str)
            df.drop(columns=["category_name"], inplace=True)

        self._collapse_rare_values("brand_name", 5000)
        self._collapse_rare_values("main_cat", 1000)
        self._collapse_rare_values("sub_cat", 1000)
        self._collapse_rare_values("sub_sub_cat", 1000)

        self._basic_features()

        print(f"✅ Loaded: Train={self.train.shape}, Test={self.test.shape}")


    def _stratified_sample(self, frac=0.35, bins=10):
        self.train["price_bin"] = pd.qcut(self.train["price"], q=bins, duplicates="drop")
        self.train = self.train.groupby("price_bin", group_keys=False).apply(
            lambda x: x.sample(frac=frac, random_state=23)
        ).reset_index(drop=True)
        print(f"⚠ Stratified undersampling applied → {self.train.shape}")


    def _collapse_rare_values(self, col, top_k, rare_label="Other"):
        combined = pd.concat([self.train[col], self.test[col]])
        valid = set(combined.value_counts().index[:top_k])
        self.train[col] = self.train[col].apply(lambda x: x if x in valid else rare_label)
        self.test[col] = self.test[col].apply(lambda x: x if x in valid else rare_label)


    def _basic_features(self):
        for df in [self.train, self.test]:
            df["name_len"]     = df["name"].str.len()
            df["desc_len"]     = df["item_description"].str.len()
            df["brand_in_name"] = df.apply(lambda r: int(r["brand_name"].lower() in r["name"].lower()), axis=1)
            df["brand_in_desc"] = df.apply(lambda r: int(r["brand_name"].lower() in r["item_description"].lower()), axis=1)
            df["shipping"] = df["shipping"].astype("category")


    def vectorize_text(self):
        print("🔧 Vectorizing text... (TF-IDF + FastText Hybrid)")

        tfidf = TfidfVectorizer(max_features=40000, ngram_range=(1,2))
        tf_train = tfidf.fit_transform(self.train["item_description"])
        tf_test = tfidf.transform(self.test["item_description"])

        svd = TruncatedSVD(n_components=120, random_state=23)
        tfidf_train = svd.fit_transform(tf_train)
        tfidf_test = svd.transform(tf_test)

        sentences = [t.split() for t in pd.concat([self.train["item_description"], self.test["item_description"]])]
        ft = FastText(sentences, vector_size=70, min_count=2, workers=4)

        def ft_vec(text):
            w = text.split()
            vecs = [ft.wv[x] for x in w if x in ft.wv]
            return np.mean(vecs, axis=0) if vecs else np.zeros(70)

        fast_train = np.vstack(self.train["item_description"].apply(ft_vec))
        fast_test = np.vstack(self.test["item_description"].apply(ft_vec))

        self.train_vectorized = pd.DataFrame(np.hstack([tfidf_train, fast_train]))
        self.test_vectorized = pd.DataFrame(np.hstack([tfidf_test, fast_test]))

        for col in ["main_cat", "sub_cat", "sub_sub_cat", "brand_name", "shipping"]:
            self.train_vectorized[col] = self.train[col].reset_index(drop=True)
            self.test_vectorized[col] = self.test[col].reset_index(drop=True)

        for col in ["name_len", "desc_len", "brand_in_name", "brand_in_desc"]:
            self.train_vectorized[col] = self.train[col].values
            self.test_vectorized[col] = self.test[col].values

        print(f"📌 Final Feature Count: {self.train_vectorized.shape[1]}")


    def setup_pycaret(self, fold=3, use_gpu=True):
        print("🔧 PyCaret setup 시작...")

        from sklearn.metrics import make_scorer
        def rmsle_func(y_true, y_pred):
            return np.sqrt(mean_squared_error(y_true, y_pred))

        rmsle_metric = make_scorer(rmsle_func, greater_is_better=False)

        self.setup_result = setup(
            data=self.train_vectorized.assign(price=self.train["price"]),
            target="price",
            fold_strategy="kfold",
            fold=fold,
            normalize=True,
            session_id=23,
            use_gpu=use_gpu,
            html=False,
            verbose=False
        )

        # ⭐ 이미 등록돼 있지 않은 경우에만 add_metric 실행
        current_metrics = get_metrics().index.tolist()
        if "rmsle" not in current_metrics:
            add_metric("rmsle", "RMSLE", rmsle_metric)

        print("✅ PyCaret setup completed.")



    def find_and_blend_models(self):
        print("🤖 Training candidate models...")

        models = ["lightgbm", "catboost", "xgboost", "ridge", "et"]
        trained = [create_model(m) for m in models]

        print("\n🍹 Blending models...")
        self.best_model = blend_models(trained, optimize="rmsle", choose_better=True)
        print("🏆 Best model selected!")


    def save_metrics(self):
        pred = predict_model(self.best_model, data=self.train_vectorized)

        y_true = np.expm1(self.train["price"])
        y_pred = np.expm1(pred["prediction_label"])

        self.metrics = {
            "RMSLE": round(np.sqrt(mean_squared_log_error(y_true, y_pred)), 4),
            "RMSE":  round(mean_squared_error(y_true, y_pred, squared=False), 4),
            "R2":    round(r2_score(y_true, y_pred), 4),
            "MAE":   round(mean_absolute_error(y_true, y_pred), 4)
        }

        fname = self._safe_name(str(self.best_model).split("(")[0])
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        path = os.path.join(self.results_dir, f"{fname}_metrics_{timestamp}.json")
        json.dump(self.metrics, open(path, "w"), indent=4)

        print(f"💾 Metrics saved → {path}")
        print(self.metrics)


    def save_best_model(self, name=None):
        name = self._safe_name(name or str(self.best_model).split("(")[0])
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        version = os.path.join(self.model_dir, f"{name}_{timestamp}")
        latest  = os.path.join(self.model_dir, f"{name}_latest")

        save_model(self.best_model, version)
        save_model(self.best_model, latest)

        print(f"💾 Saved: {version}.pkl")
        print(f"💾 Saved latest: {latest}.pkl")


    def predict_test(self):
        pred = predict_model(self.best_model, data=self.test_vectorized)
        prices = np.clip(np.expm1(pred["prediction_label"]), 0, None)

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        path = os.patһ.join(self.results_dir, f"submission_{timestamp}.csv")

        pd.DataFrame({"test_id": self.test["test_id"], "price": prices}).to_csv(path, index=False)

        print(f"📤 Submission saved → {path}")
        print(f"💡 Price range: ${prices.min():.2f} ~ ${prices.max():.2f}")


In [4]:
print("=" * 60)
print("Mercari Price Suggestion - v14 최종")
print("=" * 60)

analyzer = MercariPyCaretAnalyzer()
analyzer.load_data(undersample_frac=0.35)

Mercari Price Suggestion - v14 최종
📂 Loading dataset...
⚠ Stratified undersampling applied → (518582, 9)
✅ Loaded: Train=(518582, 15), Test=(693359, 13)


In [5]:
analyzer.vectorize_text()   # FASTTEXT + TF-IDF

🔧 Vectorizing text... (TF-IDF + FastText Hybrid)
📌 Final Feature Count: 199


In [6]:
analyzer.setup_pycaret(fold=3, use_gpu=True)
print("\n✅ 완료!")

🔧 PyCaret setup 시작...
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 2, number of used features: 0
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce GTX 1660 SUPER, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 16 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Warning] GPU acceleration is disabled because no non-trivial dense features can be found
[LightGBM] [Info] Start training from score 0.500000
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped trainin

In [7]:
analyzer.find_and_blend_models()
print("\n✅ 완료!")

🤖 Training candidate models...


         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.4229  0.3062  0.5534  0.4489  0.1377  0.1503
1     0.4180  0.3009  0.5485  0.4611  0.1364  0.1482
2     0.4210  0.3054  0.5526  0.4541  0.1373  0.1492
Mean  0.4207  0.3042  0.5515  0.4547  0.1371  0.1493
Std   0.0020  0.0024  0.0021  0.0050  0.0006  0.0009


         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.4130  0.2934  0.5416  0.4721  0.1348  0.1466
1     0.4092  0.2889  0.5375  0.4825  0.1337  0.1450
2     0.4123  0.2937  0.5420  0.4749  0.1345  0.1459
Mean  0.4115  0.2920  0.5404  0.4765  0.1343  0.1458
Std   0.0017  0.0022  0.0020  0.0044  0.0005  0.0007


         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.4163  0.2982  0.5461  0.4633  0.1358  0.1475
1     0.4120  0.2933  0.5416  0.4746  0.1347  0.1458
2     0.4155  0.2982  0.5461  0.4669  0.1356  0.1468
Mean  0.4146  0.2966  0.5446  0.4683  0.1354  0.1467
Std   0.0019  0.0023  0.0021  0.0047  0.0005  0.0007


         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.4401  0.3289  0.5735  0.4082  0.1428  0.1567
1     0.4362  0.3239  0.5691  0.4198  0.1414  0.1547
2     0.4395  0.3293  0.5739  0.4113  0.1425  0.1558
Mean  0.4386  0.3274  0.5722  0.4131  0.1422  0.1557
Std   0.0017  0.0024  0.0021  0.0049  0.0006  0.0008


         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.4128  0.2997  0.5475  0.4607  0.1364  0.1469
1     0.4086  0.2952  0.5433  0.4713  0.1351  0.1450
2     0.4113  0.3003  0.5480  0.4633  0.1360  0.1458
Mean  0.4109  0.2984  0.5462  0.4651  0.1359  0.1459
Std   0.0017  0.0023  0.0021  0.0045  0.0005  0.0008

🍹 Blending models...


         MAE     MSE    RMSE      R2   RMSLE    MAPE
Fold                                                
0     0.4124  0.2926  0.5410  0.4734  0.1346  0.1467
1     0.4078  0.2878  0.5364  0.4845  0.1334  0.1447
2     0.4112  0.2927  0.5411  0.4767  0.1343  0.1457
Mean  0.4105  0.2910  0.5395  0.4782  0.1341  0.1457
Std   0.0019  0.0023  0.0022  0.0047  0.0005  0.0008
🏆 Best model selected!

✅ 완료!


In [8]:
analyzer.save_metrics()


💾 Metrics saved → ../results\VotingRegressor_metrics_20251206_135857.json
{'RMSLE': 0.4605, 'RMSE': 29.9532, 'R2': 0.4, 'MAE': 10.6044}


In [9]:
analyzer.predict_test()
print("\n✅ 완료!")

AttributeError: module 'os' has no attribute 'patһ'

In [ ]:
# 성능 비교 루프
methods = ["tfidf", "fasttext", "bert"]
results = []

for m in methods:
    print("\n" + "="*60)
    print(f"▶ {m.upper()} 방식 실행")
    print("="*60)

    analyzer = MercariPyCaretAnalyzer()
    analyzer.load_data(undersample_frac=0.35)

    # 벡터화 (저장/불러오기 자동 처리)
    analyzer.vectorize_text(method=m)

    # PyCaret 환경 설정
    analyzer.setup_pycaret(fold=3,use_gpu=True)

    # 모델 학습 및 블렌딩
    analyzer.find_and_blend_models(use_kaggle_winners=True)

    # 성능 지표 저장
    analyzer.save_metrics(model_name=m)

    # 결과 기록
    results.append({
        "Method": m,
        "R2": analyzer.metrics["R2"],
        "RMSE": analyzer.metrics["RMSE"],
        "MAE": analyzer.metrics["MAE"]
    })

# 결과 테이블 출력
results_df = pd.DataFrame(results)
print("\n📊 성능 비교 결과")
print(results_df)